## **Connect Colab**

In [1]:
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "real-video"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "KEY"

## **Download DATASET**

In [3]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

Mounted at /content/drive
Drive mounted.
Using Colab cache for faster access to the 'ucf101-action-recognition' dataset.
ULAL_DATASET_ROOT = /kaggle/input/ucf101-action-recognition/
ULAL_OUTPUT_ROOT  = /kaggle/working/ucf101-processed/
ULAL_DATA_ROOT     = /content/UCF101
ULAL_DRIVE_PROJECT = /content/drive/MyDrive/apai
Setup complete — no config.py edit, no kernel restart.


In [4]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

ULAL_ROOT : /content/ulal
Device    : cuda
Classes   : base=10 task1=5 task2=5


42

## **Setup**

In [ ]:
n_base       = 20                                            # the student already knows 20 classes
BASE_CLASSES = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2   # SAME order the student was distilled on
META_CLASSES = cfg.SPORTS_META                              # 30 sports MAML meta-trains on (novel)
EXAM_CLASSES = cfg.SPORTS_EXAM                              # 20 NEW held-out sports to adapt to + test (50/50 with base)
n_new        = len(EXAM_CLASSES)

TRAIN_CAP = 12                                # clips/class kept for TRAIN
TEST_CAP  = 10                                # cap for the META pool only; base/exam test are FULL

# MAML meta-training: only the last few backbone blocks (+ LSTM + head) are meta-trained, so base
# features are kept; the head is re-fit afterward (recalibrate_head).
N_WAY = 5; K_SHOT = 3; K_QUERY = 3
POOL_PER_CLASS = 6
META_EPOCHS = 6; META_EPISODES = 40; INNER_STEPS = 5; META_INNER_LR = 0.05; META_LR = 1e-4
META_TRAIN_LAST = 4
HEAD_RECAL_EPOCHS = 3; HEAD_RECAL_LR = 1e-3

# Adaptation: every method trains ADAPT_EPOCHS epochs on the exam-train set (the ONE adaptation for all arms)
ADAPT_EPOCHS = 3                              # epochs of adaptation on the few-shot support
ADAPT_LR     = 3e-4                           # Adam lr during adaptation
K_MAIN = 5                                    # clips/class every method adapts on (fair few-shot)
REPLAY_PER_CLASS = 8                          # base exemplars/class in the replay buffer
FAIR_LWF = 0.5; FAIR_T = 2.0                  # LwF weight + temperature for the +LwF arms

STUDENT_PATH = f"{cfg.CKPT_DIR}/student.pt"
BASE_ROOT = f"{cfg.DATA_ROOT}/grouped_sbase"
META_ROOT = f"{cfg.DATA_ROOT}/grouped_meta"
EXAM_ROOT = f"{cfg.DATA_ROOT}/grouped_exam"

ALL_CLASSES_LIST = BASE_CLASSES + EXAM_CLASSES + META_CLASSES   # base 0-19, exam 20-39, meta 40-69
class_to_idx = {c: i for i, c in enumerate(ALL_CLASSES_LIST)}
print(f"base {n_base} | meta-train {len(META_CLASSES)} | exam {n_new} | 50/50 new/old = {n_new == n_base}")

## **Preprocess data**

In [6]:
base_split = build_group_split(BASE_CLASSES, train_groups=18, val_groups=3)
meta_split = build_group_split(META_CLASSES, train_groups=18, val_groups=3)
exam_split = build_group_split(EXAM_CLASSES, train_groups=18, val_groups=3)

In [7]:
# BASE (the 20 classes the student knows; base test for retention + replay exemplars)
preprocess_group_split(base_split, BASE_CLASSES, output_root=BASE_ROOT, max_samples=TRAIN_CAP, max_test_samples=None)


--- Preprocessing (group-disjoint) ---
To  : /content/UCF101/grouped_sbase
Classes: 20 | Train limit: 12

Processing split: train



Processing split: val



Processing split: test



Done. Clips ready at: /content/UCF101/grouped_sbase


In [ ]:
# META (30 novel SPORTS MAML meta-trains on) - test stays capped (its clips go into a RAM pool)
preprocess_group_split(meta_split, META_CLASSES, output_root=META_ROOT, max_samples=TRAIN_CAP, max_test_samples=TEST_CAP)

In [ ]:
# EXAM (20 new held-out SPORTS to adapt to and test)
preprocess_group_split(exam_split, EXAM_CLASSES, output_root=EXAM_ROOT, max_samples=TRAIN_CAP, max_test_samples=None)

## **Create Loaders**

In [10]:
base_train = UCF101Clips(os.path.join(BASE_ROOT, "train"), class_to_idx=class_to_idx)
base_test  = UCF101Clips(os.path.join(BASE_ROOT, "test"),  class_to_idx=class_to_idx)
meta_train = UCF101Clips(os.path.join(META_ROOT,     "train"), class_to_idx=class_to_idx)
exam_train = UCF101Clips(os.path.join(EXAM_ROOT,     "train"), class_to_idx=class_to_idx)
exam_test  = UCF101Clips(os.path.join(EXAM_ROOT,     "test"),  class_to_idx=class_to_idx)

base_train_loader = DataLoader(base_train, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
base_test_loader  = DataLoader(base_test,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
exam_train_loader = DataLoader(exam_train, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
exam_test_loader  = DataLoader(exam_test,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

pool_meta_clips, pool_meta_labels = build_pool([meta_train], per_class=POOL_PER_CLASS)
meta_ids = sorted(set(pool_meta_labels.tolist()))
print(f"exam train {len(exam_train)} test {len(exam_test)} | meta pool {tuple(pool_meta_clips.shape)} classes {len(meta_ids)}")

exam train 120 test 194 | meta pool (240, 16, 3, 224, 224) classes 40


## **Load the MobileNet student (knows the 20 base classes)**

In [11]:
set_seed(cfg.SEED)
base_model = MobileNetV3SmallLSTMStudent(num_classes=n_base).to(device)
base_model.load_state_dict(torch.load(STUDENT_PATH, map_location=device))
base_head  = copy.deepcopy(base_model.fc)
old_acc_before, _, _ = test_model(base_model, base_test_loader, device)
print(f"base accuracy before adapting: {old_acc_before:.4f}")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 91.1MB/s]
                                                         

Test Accuracy: 0.7585
base accuracy before adapting: 0.7585


## **MAML meta-training on the 30 meta sports**

In [12]:
set_seed(cfg.SEED)
meta_model = copy.deepcopy(base_model)
meta_model.fc = nn.Linear(cfg.HIDDEN_SIZE, N_WAY).to(device)
meta_model, hist_maml = meta_train_maml(meta_model, pool_meta_clips, pool_meta_labels, meta_ids,
                                        N_WAY, K_SHOT, K_QUERY, META_EPOCHS, META_EPISODES,
                                        META_INNER_LR, INNER_STEPS, META_LR, device,
                                        freeze_backbone=True, train_last=META_TRAIN_LAST)
meta_model = recalibrate_head(meta_model, base_train_loader, n_base, device, HEAD_RECAL_EPOCHS, HEAD_RECAL_LR)
base_after_meta, _, _ = test_model(meta_model, base_test_loader, device)
print(f"meta init base accuracy after head recalibration: {base_after_meta:.4f}")
del pool_meta_clips, pool_meta_labels
gc.collect(); torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/rnn.py:1169: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1479.)
  result = _VF.lstm(


[maml] epoch 1/6 | meta query acc 0.8483
[maml] epoch 2/6 | meta query acc 0.8833
[maml] epoch 3/6 | meta query acc 0.9150
[maml] epoch 4/6 | meta query acc 0.9233
[maml] epoch 5/6 | meta query acc 0.9283
[maml] epoch 6/6 | meta query acc 0.9467


Test Accuracy: 0.6902
meta init base accuracy after head recalibration: 0.6902


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, META_EPOCHS + 1), hist_maml["query_accs"], marker="o")
plt.xlabel("meta epoch"); plt.ylabel("meta query accuracy"); plt.ylim(0, 1.05)
plt.title("MAML meta-training on the sports pool")
plt.show()

## **Shared replay buffer + LwF teacher**

In [14]:
base_buffer = ReplayBuffer(max_size=REPLAY_PER_CLASS * n_base + 10)
fill_buffer(base_buffer, base_train_loader, per_class=REPLAY_PER_CLASS)
base_teacher = snapshot_teacher(base_model)   # 20-way teacher, protects the base classes via LwF
print(f"replay buffer: {len(base_buffer.data)} base clips")

replay buffer: 160 base clips


## **Adapt each method (fair few-shot)**

Every method adapts the student to the 20 new classes on the SAME support - `K_MAIN` clips per class,
chosen deterministically - for the SAME `ADAPT_EPOCHS` epochs, same lr, same seed, and the same replay
buffer / LwF teacher. Only two things change between methods: the init (plain student vs the MAML-meta
init) and whether replay / LwF are switched on. That is what makes it a fair comparison. Each method has
its own section below with its confusion matrix over all 40 classes.

In [ ]:
# combined test (for the confusion matrices) and the fair few-shot support
cm_classes  = BASE_CLASSES + EXAM_CLASSES
cm_loader   = DataLoader(ConcatDataset([base_test, exam_test]), batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
n_base_test = len(base_test)
n_new_test  = len(exam_test)

# K_MAIN clips per new class, deterministic -> every method sees the identical support (fair)
sup_x, sup_y = build_pool([exam_train], per_class=K_MAIN)
main_loader  = DataLoader(TensorDataset(sup_x, sup_y), batch_size=cfg.BATCH_SIZE, shuffle=True)

arms = [
    ("no-meta",         base_model, None,        None),
    ("replay",          base_model, base_buffer, None),
    ("replay+LwF",      base_model, base_buffer, base_teacher),
    ("MAML",            meta_model, None,        None),
    ("MAML+replay",     meta_model, base_buffer, None),
    ("MAML+replay+LwF", meta_model, base_buffer, base_teacher),
]
models = {}
curves = {}
curves_base = {}
RESULTS = []
print(f"{n_base} base + {n_new} new | adapt on {K_MAIN} clips/class for {ADAPT_EPOCHS} epochs | base before {old_acc_before:.3f}")

## **no-meta**

In [ ]:
set_seed(cfg.SEED)
m, new_acc, base_acc, nc, bc = adapt_and_eval(copy.deepcopy(base_model), main_loader, base_test_loader,
                                              exam_test_loader, n_base, n_new, device,
                                              buffer=None, teacher=None,
                                              epochs=ADAPT_EPOCHS, lr=ADAPT_LR, lambda_distill=FAIR_LWF, T=FAIR_T)
overall = (new_acc * n_new_test + base_acc * n_base_test) / (n_new_test + n_base_test)
models["no-meta"] = m
curves["no-meta"] = nc
curves_base["no-meta"] = bc
RESULTS.append({"method": "no-meta", "new_acc": round(new_acc, 3),
                "base_acc": round(base_acc, 3), "overall_acc": round(overall, 3)})
print(f"no-meta: new {new_acc:.3f} | base {base_acc:.3f} | overall {overall:.3f}")
_, preds, labels = test_model(m, cm_loader, device)
plot_confusion_matrix(labels, preds, num_classes=n_base + n_new, classes_list=cm_classes, name="no-meta")
torch.cuda.empty_cache()

## **replay**

In [ ]:
set_seed(cfg.SEED)
m, new_acc, base_acc, nc, bc = adapt_and_eval(copy.deepcopy(base_model), main_loader, base_test_loader,
                                              exam_test_loader, n_base, n_new, device,
                                              buffer=base_buffer, teacher=None,
                                              epochs=ADAPT_EPOCHS, lr=ADAPT_LR, lambda_distill=FAIR_LWF, T=FAIR_T)
overall = (new_acc * n_new_test + base_acc * n_base_test) / (n_new_test + n_base_test)
models["replay"] = m
curves["replay"] = nc
curves_base["replay"] = bc
RESULTS.append({"method": "replay", "new_acc": round(new_acc, 3),
                "base_acc": round(base_acc, 3), "overall_acc": round(overall, 3)})
print(f"replay: new {new_acc:.3f} | base {base_acc:.3f} | overall {overall:.3f}")
_, preds, labels = test_model(m, cm_loader, device)
plot_confusion_matrix(labels, preds, num_classes=n_base + n_new, classes_list=cm_classes, name="replay")
torch.cuda.empty_cache()

## **replay+LwF**

In [ ]:
set_seed(cfg.SEED)
m, new_acc, base_acc, nc, bc = adapt_and_eval(copy.deepcopy(base_model), main_loader, base_test_loader,
                                              exam_test_loader, n_base, n_new, device,
                                              buffer=base_buffer, teacher=base_teacher,
                                              epochs=ADAPT_EPOCHS, lr=ADAPT_LR, lambda_distill=FAIR_LWF, T=FAIR_T)
overall = (new_acc * n_new_test + base_acc * n_base_test) / (n_new_test + n_base_test)
models["replay+LwF"] = m
curves["replay+LwF"] = nc
curves_base["replay+LwF"] = bc
RESULTS.append({"method": "replay+LwF", "new_acc": round(new_acc, 3),
                "base_acc": round(base_acc, 3), "overall_acc": round(overall, 3)})
print(f"replay+LwF: new {new_acc:.3f} | base {base_acc:.3f} | overall {overall:.3f}")
_, preds, labels = test_model(m, cm_loader, device)
plot_confusion_matrix(labels, preds, num_classes=n_base + n_new, classes_list=cm_classes, name="replay+LwF")
torch.cuda.empty_cache()

## **MAML**

In [ ]:
set_seed(cfg.SEED)
m, new_acc, base_acc, nc, bc = adapt_and_eval(copy.deepcopy(meta_model), main_loader, base_test_loader,
                                              exam_test_loader, n_base, n_new, device,
                                              buffer=None, teacher=None,
                                              epochs=ADAPT_EPOCHS, lr=ADAPT_LR, lambda_distill=FAIR_LWF, T=FAIR_T)
overall = (new_acc * n_new_test + base_acc * n_base_test) / (n_new_test + n_base_test)
models["MAML"] = m
curves["MAML"] = nc
curves_base["MAML"] = bc
RESULTS.append({"method": "MAML", "new_acc": round(new_acc, 3),
                "base_acc": round(base_acc, 3), "overall_acc": round(overall, 3)})
print(f"MAML: new {new_acc:.3f} | base {base_acc:.3f} | overall {overall:.3f}")
_, preds, labels = test_model(m, cm_loader, device)
plot_confusion_matrix(labels, preds, num_classes=n_base + n_new, classes_list=cm_classes, name="MAML")
torch.cuda.empty_cache()

## **MAML+replay**

In [ ]:
set_seed(cfg.SEED)
m, new_acc, base_acc, nc, bc = adapt_and_eval(copy.deepcopy(meta_model), main_loader, base_test_loader,
                                              exam_test_loader, n_base, n_new, device,
                                              buffer=base_buffer, teacher=None,
                                              epochs=ADAPT_EPOCHS, lr=ADAPT_LR, lambda_distill=FAIR_LWF, T=FAIR_T)
overall = (new_acc * n_new_test + base_acc * n_base_test) / (n_new_test + n_base_test)
models["MAML+replay"] = m
curves["MAML+replay"] = nc
curves_base["MAML+replay"] = bc
RESULTS.append({"method": "MAML+replay", "new_acc": round(new_acc, 3),
                "base_acc": round(base_acc, 3), "overall_acc": round(overall, 3)})
print(f"MAML+replay: new {new_acc:.3f} | base {base_acc:.3f} | overall {overall:.3f}")
_, preds, labels = test_model(m, cm_loader, device)
plot_confusion_matrix(labels, preds, num_classes=n_base + n_new, classes_list=cm_classes, name="MAML+replay")
torch.cuda.empty_cache()

## **MAML+replay+LwF**

In [ ]:
set_seed(cfg.SEED)
m, new_acc, base_acc, nc, bc = adapt_and_eval(copy.deepcopy(meta_model), main_loader, base_test_loader,
                                              exam_test_loader, n_base, n_new, device,
                                              buffer=base_buffer, teacher=base_teacher,
                                              epochs=ADAPT_EPOCHS, lr=ADAPT_LR, lambda_distill=FAIR_LWF, T=FAIR_T)
overall = (new_acc * n_new_test + base_acc * n_base_test) / (n_new_test + n_base_test)
models["MAML+replay+LwF"] = m
curves["MAML+replay+LwF"] = nc
curves_base["MAML+replay+LwF"] = bc
RESULTS.append({"method": "MAML+replay+LwF", "new_acc": round(new_acc, 3),
                "base_acc": round(base_acc, 3), "overall_acc": round(overall, 3)})
print(f"MAML+replay+LwF: new {new_acc:.3f} | base {base_acc:.3f} | overall {overall:.3f}")
_, preds, labels = test_model(m, cm_loader, device)
plot_confusion_matrix(labels, preds, num_classes=n_base + n_new, classes_list=cm_classes, name="MAML+replay+LwF")
torch.cuda.empty_cache()

## **Adaptation curves**

New-class accuracy per epoch (left) and base retention per epoch (right). Steeper new-class line = learns
faster; flat, high base line = forgets less.

In [ ]:
plt.figure(figsize=(9, 5))
for name in curves:
    plt.plot(range(len(curves[name])), curves[name], marker="o", label=name)
plt.xlabel("epoch (0 = before adapting)"); plt.ylabel("new-class accuracy"); plt.ylim(0, 1.05)
plt.title(f"How each method learns the new classes ({K_MAIN}-shot)"); plt.legend(); plt.show()

plt.figure(figsize=(9, 5))
for name in curves_base:
    plt.plot(range(len(curves_base[name])), curves_base[name], marker="o", label=name)
plt.axhline(old_acc_before, ls="--", color="gray", label="base before")
plt.xlabel("epoch (0 = before adapting)"); plt.ylabel("base accuracy"); plt.ylim(0, 1.05)
plt.title("How much base knowledge each method keeps"); plt.legend(); plt.show()

## **Compare all methods (K_MAIN-shot)**

In [ ]:
compare_df = pd.DataFrame(RESULTS).sort_values("overall_acc", ascending=False).reset_index(drop=True)
print(f"{K_MAIN}-shot, {ADAPT_EPOCHS} epochs (base before adapting {old_acc_before:.3f}):")
print(compare_df.to_string(index=False))

x = np.arange(len(compare_df))
w = 0.4
plt.figure(figsize=(10, 5))
plt.bar(x - w / 2, compare_df["new_acc"],  w, label="new")
plt.bar(x + w / 2, compare_df["base_acc"], w, label="base")
plt.axhline(old_acc_before, ls="--", color="gray", label="base before")
plt.xticks(x, compare_df["method"], rotation=20, ha="right"); plt.ylim(0, 1.05)
plt.ylabel("accuracy"); plt.title(f"New vs base accuracy per method ({K_MAIN}-shot)"); plt.legend(); plt.show()

## **Prototype test (meta features)**

Did meta-training reshape the features? Nearest-class-mean accuracy on the meta classes, plain student vs
meta-trained one (features only, no head).

In [ ]:
meta_train_loader = DataLoader(meta_train, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
meta_test        = UCF101Clips(os.path.join(META_ROOT, "test"), class_to_idx=class_to_idx)
meta_test_loader = DataLoader(meta_test, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

plain_proto = proto_accuracy(base_model, meta_train_loader, meta_test_loader, device)
meta_proto  = proto_accuracy(meta_model, meta_train_loader, meta_test_loader, device)
print(f"prototype accuracy on the meta classes: plain {plain_proto:.3f} | meta-trained {meta_proto:.3f}")

## **Final scores table**

In [ ]:
rows = []
for name in models:
    _, preds, labels = test_model(models[name], cm_loader, device)
    preds = np.array(preds); labels = np.array(labels)
    base_mask = labels < n_base
    new_mask  = labels >= n_base
    p, r, f, _ = precision_recall_fscore_support(labels, preds, labels=list(range(n_base + n_new)),
                                                 average=None, zero_division=0)
    rows.append({"method": name,
                 "new_acc":  round(float((preds[new_mask]  == labels[new_mask]).mean()),  3),
                 "base_acc": round(float((preds[base_mask] == labels[base_mask]).mean()), 3),
                 "new_f1":   round(float(f[n_base:].mean()), 3),
                 "base_f1":  round(float(f[:n_base].mean()), 3)})
scores_df = pd.DataFrame(rows).sort_values("new_f1", ascending=False).reset_index(drop=True)
print(scores_df.to_string(index=False))

## **Save**

In [ ]:
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
payload = {"n_base": n_base, "n_new": n_new, "n_meta": len(META_CLASSES),
           "adapt_epochs": ADAPT_EPOCHS, "k_main": K_MAIN,
           "old_acc_before": old_acc_before,
           "results": RESULTS,
           "proto": {"plain": plain_proto, "meta_trained": meta_proto},
           "maml_curve": hist_maml["query_accs"]}
with open(f"{cfg.RESULTS_DIR}/stage5_meta_vs_rehearsal.json", "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved -> {cfg.RESULTS_DIR}/stage5_meta_vs_rehearsal.json")